# Lab 10 — MLlib: ajuste, persistência e scoring em streaming

> Extraído de `aulas/Aula_7_mllib.ipynb` (seções 7 e 8 e resumo). As referências a números de seção usam a numeração do notebook original.

## Preparação

Este notebook roda sozinho: as células abaixo recriam o estado necessário, vindo da primeira parte.

In [ ]:
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.functions import vector_to_array

spark = (
    SparkSession.builder
    .appName("Aula 7 - MLlib")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
N = 200_000
REGIONS = ["North", "Northeast", "Midwest", "Southeast", "South"]

def pick(values, seed):
    arr = F.array(*[F.lit(v) for v in values])
    return F.element_at(arr, (F.rand(seed) * len(values)).cast("int") + 1)

base = (
    spark.range(N).withColumnRenamed("id", "customer_id")
    .withColumn("tenure_months",   (F.rand(1) * 71 + 1).cast("int"))
    .withColumn("plan",            pick(["Basic", "Standard", "Premium"], 2))
    .withColumn("region",          pick(REGIONS, 3))
    .withColumn("monthly_spend",   F.round(
        F.when(F.col("plan") == "Basic", 40).when(F.col("plan") == "Standard", 120).otherwise(300)
        * (0.5 + F.rand(4)), 2))
    .withColumn("n_orders_90d",    (F.rand(5) * 15).cast("int"))
    .withColumn("support_tickets", (F.pow(F.rand(6), 2) * 9).cast("int"))
    .withColumn("has_discount",    F.rand(7) < 0.3)
    .withColumn("last_rating",     (F.rand(8) * 5 + 1).cast("int"))
)

logit = (
    0.8
    - 0.045 * F.col("tenure_months")
    + 0.45  * F.col("support_tickets")
    - 0.10  * F.col("n_orders_90d")
    - 0.45  * (F.col("last_rating") - 3)
    + F.when(F.col("plan") == "Basic", 0.5).when(F.col("plan") == "Premium", -0.6).otherwise(0.0)
    - 0.5   * F.col("has_discount").cast("int")
    + 0.6   * F.randn(9)
)

customers = (
    base
    .withColumn("churn", (F.rand(10) < 1 / (1 + F.exp(-logit))).cast("int"))
    .withColumn("last_rating",   F.when(F.rand(11) < 0.06, None).otherwise(F.col("last_rating")))
    .withColumn("monthly_spend", F.when(F.rand(12) < 0.03, None).otherwise(F.col("monthly_spend")))
)

customers.write.mode("overwrite").parquet("/tmp/aula7/customers")
customers = spark.read.parquet("/tmp/aula7/customers")
customers.show(5)

In [ ]:
train, test = customers.randomSplit([0.8, 0.2], seed=42)
train.cache(); test.cache()
print(f"Treino: {train.count():,} | Teste: {test.count():,}")

In [ ]:
sample = train.limit(5)

imputer = Imputer(strategy="median",
                  inputCols=["last_rating", "monthly_spend"],
                  outputCols=["last_rating_imp", "monthly_spend_imp"])
step1 = imputer.fit(train).transform(sample)
step1.select("last_rating", "last_rating_imp", "monthly_spend", "monthly_spend_imp").show()

In [ ]:
indexer = StringIndexer(inputCols=["plan", "region"], outputCols=["plan_idx", "region_idx"],
                        handleInvalid="keep")          # categoria nova no futuro -> índice extra
encoder = OneHotEncoder(inputCols=["plan_idx", "region_idx"], outputCols=["plan_vec", "region_vec"])

idx_model = indexer.fit(train)
print("Ordem das categorias (da mais frequente para a menos):", idx_model.labelsArray)

step2 = encoder.fit(idx_model.transform(train)).transform(idx_model.transform(sample))
step2.select("plan", "plan_idx", "plan_vec", "region", "region_vec").show(truncate=False)

In [ ]:
NUMERIC = ["tenure_months", "monthly_spend_imp", "n_orders_90d", "support_tickets", "last_rating_imp"]

assembler = VectorAssembler(inputCols=NUMERIC + ["has_discount", "plan_vec", "region_vec"],
                            outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=False)

prep = Pipeline(stages=[imputer, indexer, encoder, assembler, scaler]).fit(train)
prep.transform(sample).select("customer_id", "features").show(3, truncate=False)

In [ ]:
lr = LogisticRegression(featuresCol="features", labelCol="churn", maxIter=50)
lr_pipeline = Pipeline(stages=[imputer, indexer, encoder, assembler, scaler, lr])

t0 = time.perf_counter()
lr_model = lr_pipeline.fit(train)
print(f"Treino: {time.perf_counter() - t0:.1f} s")

pred = lr_model.transform(test)
pred.select("customer_id", "churn", "probability", "prediction").show(5, truncate=False)

In [ ]:
def evaluate(predictions, name="modelo"):
    auc_roc = BinaryClassificationEvaluator(labelCol="churn", metricName="areaUnderROC").evaluate(predictions)
    auc_pr  = BinaryClassificationEvaluator(labelCol="churn", metricName="areaUnderPR").evaluate(predictions)
    mc = lambda m, **kw: MulticlassClassificationEvaluator(labelCol="churn", metricName=m, **kw).evaluate(predictions)
    print(f"{name:<22} AUC-ROC={auc_roc:.3f}  AUC-PR={auc_pr:.3f}  "
          f"accuracy={mc('accuracy'):.3f}  precision={mc('precisionByLabel', metricLabel=1.0):.3f}  "
          f"recall={mc('recallByLabel', metricLabel=1.0):.3f}")

evaluate(pred, "Regressão logística")

print("\nMatriz de confusão:")
pred.groupBy("churn").pivot("prediction", [0.0, 1.0]).count().orderBy("churn").show()

---
## 7. Ajuste de hiperparâmetros com validação cruzada

```
  treino dividido em k = 3 partes (folds)

  combinação de parâmetros 1:  [treina ][treina ][valida ]  -> AUC_1
                               [treina ][valida ][treina ]  -> AUC_2
                               [valida ][treina ][treina ]  -> AUC_3   => média

  combinação de parâmetros 2:  ...                                    => média

  a melhor média é retreinada com o treino inteiro; o TESTE só é usado no final
```

Custo: `combinações x folds` treinos. O parâmetro `parallelism` treina várias combinações ao mesmo tempo.

In [ ]:
grid = (ParamGridBuilder()
        .addGrid(lr.regParam, [0.0, 0.01, 0.1])
        .addGrid(lr.elasticNetParam, [0.0, 1.0])     # 0 = ridge (L2), 1 = lasso (L1)
        .build())

cv = CrossValidator(
    estimator=lr_pipeline,
    estimatorParamMaps=grid,
    evaluator=BinaryClassificationEvaluator(labelCol="churn", metricName="areaUnderROC"),
    numFolds=3,
    parallelism=2,
    seed=42,
)

t0 = time.perf_counter()
cv_model = cv.fit(train)
print(f"{len(grid)} combinações x 3 folds: {time.perf_counter() - t0:.1f} s\n")

for params, metric in sorted(zip(grid, cv_model.avgMetrics), key=lambda x: -x[1]):
    print(f"AUC média = {metric:.4f} | " + ", ".join(f"{p.name}={v}" for p, v in params.items()))

best = cv_model.bestModel
evaluate(best.transform(test), "\nMelhor LR (teste)")

A regularização não ajudou: com 160 mil linhas e 14 features, não há overfitting para combater. Ela faz diferença com muitas features, poucos dados ou features correlacionadas. O processo, porém, é o mesmo para qualquer modelo e grade de parâmetros.

---
## 8. Salvar, carregar e usar o modelo

O `PipelineModel` salvo contém **todos os estágios**: medianas do Imputer, categorias do StringIndexer, escalas e coeficientes. Quem carrega o modelo aplica exatamente a mesma preparação, sem reescrever código.

In [ ]:
MODEL_PATH = "/tmp/aula7/models/churn_lr"
best.write().overwrite().save(MODEL_PATH)

loaded = PipelineModel.load(MODEL_PATH)
print("Estágios:", [type(s).__name__ for s in loaded.stages])

# Novos clientes, inclusive com nulo e com uma região que não existia no treino
new_customers = spark.createDataFrame(
    [(900001, 3,  "Basic",   "South",   None,  1, 6, False, 1),
     (900002, 60, "Premium", "North",   410.0, 12, 0, True, 5),
     (900003, 12, "Standard","Islands", 95.0,  4, 2, False, None)],
    "customer_id long, tenure_months int, plan string, region string, monthly_spend double, "
    "n_orders_90d int, support_tickets int, has_discount boolean, last_rating int",
)

(loaded.transform(new_customers)
 .select("customer_id", "plan", "support_tickets",
         F.round(vector_to_array("probability")[1], 3).alias("p_churn"))
 .show())

### 8.1 Scoring em streaming

Um `PipelineModel` é um Transformer, e Transformers funcionam em DataFrames de streaming. Conectando com a aula 6: cada cliente que chega em uma pasta recebe uma pontuação de risco em segundos.

In [ ]:
import os, shutil
STREAM_IN = "/tmp/aula7/stream_in"
STAGING   = "/tmp/aula7/staging"
for d in [STREAM_IN, STAGING]:
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d)

stream = spark.readStream.schema(new_customers.schema).json(STREAM_IN)

q = (loaded.transform(stream)
     .select("customer_id", F.round(vector_to_array("probability")[1], 3).alias("p_churn"))
     .filter("p_churn >= 0.5")                                # só clientes em risco
     .writeStream.format("memory").queryName("at_risk").outputMode("append").start())

# escrita atômica (aula 6): escreve fora da pasta monitorada e depois move
with open(f"{STAGING}/batch1.json", "w") as f:
    for row in new_customers.toJSON().collect():
        f.write(row + "\n")
os.replace(f"{STAGING}/batch1.json", f"{STREAM_IN}/batch1.json")

q.processAllAvailable()
spark.sql("SELECT * FROM at_risk").show()
q.stop()

---
## Resumo

| Etapa | MLlib |
|-------|-------|
| Dividir | `df.randomSplit([0.8, 0.2], seed=42)` |
| Nulos | `Imputer(strategy="median")` |
| Categorias | `StringIndexer(handleInvalid="keep")` + `OneHotEncoder` |
| Vetor de features | `VectorAssembler` |
| Escala | `StandardScaler` |
| Features derivadas no pipeline | `SQLTransformer` |
| Classificação | `LogisticRegression`, `RandomForestClassifier`, `GBTClassifier` |
| Avaliação | `BinaryClassificationEvaluator`, `MulticlassClassificationEvaluator` |
| Tuning | `ParamGridBuilder` + `CrossValidator` |
| Probabilidade como número | `vector_to_array("probability")[1]` |
| Persistência | `model.write().overwrite().save(path)` / `PipelineModel.load(path)` |

### O curso até aqui

```
  Aula 3  API de DataFrames, Pandas vs Spark
  Aula 4  Performance: shuffles, joins, skew, AQE, cache
  Aula 5  ETL Medallion + Delta Lake
  Aula 6  Structured Streaming
  Aula 7  MLlib

  arquivos brutos ─▶ Bronze ─▶ Silver ─▶ Gold ─▶ modelo ─▶ previsões em batch e streaming
```

In [ ]:
spark.stop()